# **PROMPT**

In [20]:
# TASK - 1  [PROMPT]
def prompt_1(input_data=None):

   template_1 = '''Your task is to search for and classify 10 of the latest scientific research news articles published strictly within the last 30 days, compile them into a clean markdown report, and email that report.

Search Priority:
1. First Priority (Focused Topics): Search for recent news in Space, astronomy, novel discoveries or inventions of objects, subatomic particles, and curious phenomena.
2. Fallback Rule (Crucial): If no recent research is found matching the focused topics above, do NOT stop or return empty. Immediately pivot and search for other groundbreaking scientific research news from the past 30 days across any of the allowed categories below.

Allowed Categories:
- Space & Astronomy
- Physics & Particle Science
- Materials Science & Nanotechnology
- Robotics & Engineering
- Artificial Intelligence & Machine Learning
- Chemistry & Energy
- Healthcare & Medicine
- Other / Interdisciplinary Research

Instructions:
1. 1. Important Rules: (Mandatory 10 New Articles and sending email (8th point)):
   - Do not chat or explain what you are doing. Do not say "I will search again" or "Here are my results".
   - If you have fewer than 10 articles from the last 30 days, search again with new keywords until you have all 10.
   - Once you have all 10 articles, write the markdown report and send the email to the recipient.
   - After the email is sent, reply with "Done" and stop.
   - Only select articles published strictly within the last 30 days. Discard any older articles.
   - If any articles are discarded or you have fewer than 10, search again with different keywords across the allowed categories.
   - Continue searching until you have collected exactly 10 different articles, recent articles before creating the report.
2. If focused topics yield fewer than 10 articles, fill the remaining spots with top scientific breakthroughs from the allowed categories.
3. Classify each article with one primary category and optional secondary categories.
4. Add an AI-related tag (Yes / No).
5. For each article, provide:
   - Title
   - Publication Date
   - Source
   - Original Article Link / URL
   - Detailed Summary
   - Categories (Primary & Secondary)
   - AI Tag
6. Use only information retrieved from search—do not invent details.
7. Format all 10 articles cleanly in Markdown format.
8. Once you have compiled the report, send an email with the research report to godandghostofshinobi@gmail.com from athishprajwalgr006@gmail.com with subject "Latest Scientific Research News".
'''
   return template_1

from langchain_core.runnables import RunnableLambda
prompt_1_runnable = RunnableLambda(prompt_1)






# **ReAct Agent - Tavily , GmailToolkit**

from langchain_tavily import TavilySearch
search_tool = TavilySearch(max_results=5,description='this is a tool to search the web for news', time_range='month' ,   topic="general",
    include_usage=True,search_depth='basic')


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=856541862141-tpig7hmdt8l44ftakg5rcl3j4klcb7n9.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A65227%2F&scope=https%3A%2F%2Fmail.google.com%2F&state=tmMtwcNhv9W5vumw4ztm6QAUF5Jheb&code_challenge=yxdreckeXrAQgf2LHLk72FzkMN-THpEcOLI5E98yK7o&code_challenge_method=S256&access_type=offline

def tools_kit():

    from langchain_tavily import TavilySearch
    search_tool = TavilySearch(max_results=10,description='this is a tool to search the web for news', time_range='month' ,   topic="general",
        include_usage=True,search_depth='basic')

    from langchain.tools import tool
    import resend
    import os
    from dotenv import load_dotenv
    import base64


    
    load_dotenv()
    
    tool_kit = [search_tool,send_research_email]
    return tool_kit

tools_kit_wrap = RunnableLambda(tools_kit)


def agent_creation():
    from langchain.agents import create_agent
    from langchain_ollama import ChatOllama

    llm = ChatOllama(
        model="qwen3.5:9b",
        temperature=0,
        reasoning=False,
        num_ctx=8192,
        num_predict=1024,
        keep_alive="30m"
    )

    agent = create_agent(model=llm, tools=tool_Kit)
    return agent

from langchain_core.runnables import RunnableLambda

agent_creation_wrap = RunnableLambda(agent_creation)

result = agent.invoke(
    {"messages": [{"role": "user", "content": template_1}]})
result
    

from IPython.display import display, Markdown

display(Markdown(result["messages"][-1].content))

In [21]:
import sqlite3
import re
from pathlib import Path

DB_PATH = Path("sent_papers.db")

In [22]:
import sqlite3
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

DB_PATH = Path("sent_papers.db")


def init_db():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS sent_papers (
                paper_key TEXT PRIMARY KEY,
                title TEXT,
                url TEXT,
                emailed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)


def get_paper_key(paper):
    doi = (paper.get("doi") or "").strip().lower()
    url = (paper.get("url") or "").strip()

    # Prefer DOI because URLs can change
    doi = doi.removeprefix("https://doi.org/")
    doi = doi.removeprefix("http://doi.org/")

    if doi:
        return "doi:" + doi

    if url:
        parts = urlsplit(url)
        clean_url = urlunsplit((
            parts.scheme.lower(),
            parts.netloc.lower(),
            parts.path.rstrip("/"),
            "", ""
        ))
        return "url:" + clean_url

    return "title:" + " ".join(
        (paper.get("title") or "").lower().split()
    )


def get_new_papers(papers):
    init_db()
    new_papers = []

    with sqlite3.connect(DB_PATH) as conn:
        for paper in papers:
            key = get_paper_key(paper)

            if key == "title:":
                continue

            exists = conn.execute(
                "SELECT 1 FROM sent_papers WHERE paper_key = ?",
                (key,)
            ).fetchone()

            if not exists:
                new_papers.append(paper)

    return new_papers


def mark_papers_sent(papers):
    init_db()

    with sqlite3.connect(DB_PATH) as conn:
        for paper in papers:
            conn.execute("""
                INSERT OR IGNORE INTO sent_papers
                (paper_key, title, url)
                VALUES (?, ?, ?)
            """, (
                get_paper_key(paper),
                paper.get("title", ""),
                paper.get("url", "")
            ))


init_db()
print("SQLite database ready:", DB_PATH.resolve())

SQLite database ready: C:\Users\Sampath Kumar G R\Downloads\AI RESEARCH NEWS\AI RESEARCH NEWS\sent_papers.db


In [23]:
import os, re, base64, requests
from langchain.tools import tool

URL_RE = re.compile(r'https?://[^\s\)\]>]+')

def extract_papers(text):
    """Works on the report text. A new paper starts at every Title line."""
    papers, cur = [], {}
    for raw in text.splitlines():
        line = raw.replace("**", "").strip()
        line = re.sub(r"^[-*#>\s\d.]+", "", line)      # strip bullets, #, "1."
        m = re.match(
            r"(?i)^(Title|Paper Title|DOI|Authors?|Publication Date|Date|"
            r"Original Article Link\s*/\s*URL|Article Link|Source URL|URL|Link|Source)"
            r"\s*:\s*(.+)", line)
        if not m:
            continue
        field, value = m.group(1).lower(), m.group(2).strip()

        if field in ("title", "paper title"):
            if cur.get("title"):
                papers.append(cur)
                cur = {}
            cur["title"] = value
        elif field == "doi":
            cur["doi"] = value
        elif field in ("author", "authors"):
            cur["authors"] = value
        elif field in ("date", "publication date"):
            cur["publication_date"] = value
        else:  # URL-like fields, including "Source"
            u = URL_RE.search(value)             # only accept a real link
            if u:
                cur["url"] = u.group(0).rstrip(".,")
    if cur.get("title"):
        papers.append(cur)
    return papers





In [24]:
import sqlite3
with sqlite3.connect("sent_papers.db") as c:
    c.execute("DELETE FROM sent_papers WHERE emailed_at >= '2026-11-01'")
    print("rows left:", c.execute("SELECT COUNT(*) FROM sent_papers").fetchone()[0])
    

rows left: 0


In [ ]:
from IPython.core import magic_arguments
from langchain import agents
from dotenv import load_dotenv
load_dotenv()

def prompt_1_result(template_1):
     
    # LLM Setup
    from langchain_mistralai import ChatMistralAI

    llm = ChatMistralAI(model="ministral-14b-2512", temperature=0.2, max_retries=5)
    
    # TAVILY SEARCH TOOL
    from langchain_tavily import TavilySearch
    search_tool = TavilySearch(max_results=5,description='this is a tool to search the web for news', time_range='month' ,   topic="general",
        include_usage=True,search_depth='basic')

    
    # Importing resend and creating a tool decorator in langchain (SADASEND EMAIL TOOL)

    import os, smtplib, ssl
    from email.message import EmailMessage
    from dotenv import load_dotenv
    from langchain.tools import tool
    load_dotenv()

    @tool
    def send_email(from_email: str, to_email: str, subject: str, markdown_content: str) -> str:
        """Send the research report by email. Rejects articles that were already emailed."""
        papers = extract_papers(markdown_content)
        if not papers:
            return ("REJECTED: no articles found. Write each article with lines like "
                    "'Title: ...' and 'URL: https://...'.")

        new = get_new_papers(papers)
        dupes = [p["title"] for p in papers if p not in new]
        if dupes:
            return ("REJECTED: already emailed: " + "; ".join(dupes) +
                    ". Replace ONLY these with new articles and call send_email again "
                    "with the full report.")
 
        user = os.getenv("GMAIL_USER")
        pw = (os.getenv("GMAIL_APP_PASSWORD") or "").replace(" ", "")
        if not user or not pw:
            return "Failed: GMAIL_USER or GMAIL_APP_PASSWORD not found in environment."

        msg = EmailMessage()
        msg["From"] = user
        msg["To"] = to_email
        msg["Subject"] = subject
        msg.set_content("Attached is the research news report.")
        msg.add_attachment(markdown_content.encode("utf-8"), maintype="text",
                        subtype="plain", filename="Research_News.txt")
        try:
            with smtplib.SMTP_SSL("smtp.gmail.com", 465,
                                context=ssl.create_default_context(), timeout=20) as s:
                s.login(user, pw)
                s.send_message(msg)
        except Exception as e:
            return f"Failed to send email: {e}"

        mark_papers_sent(papers)          # saved only after a successful send
        return f"Email sent. {len(papers)} articles recorded."


    #Agent Creation

    from langchain.agents import create_agent


    Tool_kit = [search_tool,send_email]
    agent = create_agent(model=llm, tools=Tool_kit)


    from IPython.display import display, Markdown

    # --- OUTPUT 1: The Research News ---
    print("Running Prompt 1: Collecting Research News...")
    result_1 = agent.invoke(
        {"messages": [{"role": "user", "content": template_1}]},
        config={"recursion_limit": 40})

    # --- debug ---
    for m in result_1["messages"]:
        for c in getattr(m, "tool_calls", None) or []:
            print("CALL:", c["name"])
        if m.type == "tool":
            print("TOOL RESULT:", m.name, "->", str(m.content)[:300])
    # -------------

    display(Markdown(result_1["messages"][-1].content))
    

    
    template_2 = """You are a scientific research matching agent.

Research Concept to Validate:
"Using magnetic fluid-like or clay-like materials with self-healing properties to capture, control the rotation of, or mitigate impacts from tiny meteoroids and space debris on spacecraft."

Specific Space Problem to Address:
"Spacecraft operating in orbital environments face catastrophic structural and operational risks from untrackable micro-debris and space dust (ranging from 1 micrometer to 1 centimetre) traveling at hypervelocities between 7 km/s and 15 km/s. Current aerospace architectures lack lightweight, high-efficiency mitigation systems capable of safely absorbing or redirecting the extreme kinetic energy of these hypervelocity impacts without causing structural punctures, permanent sensor blinding, or secondary material fragmentation that exacerbates the orbital debris field."

Target Criteria:
Search for scientific research papers (on arXiv, IEEE, ScienceDirect, NASA ADS, Nature, etc.) that discuss BOTH:
1. Magnetic materials or fluids (e.g., ferrofluids, magnetorheological fluids, magnetic elastomers) or self-healing protective materials.
AND
2. Their application in addressing the problem above: specifically mitigating hypervelocity impacts (7–15 km/s) from untrackable micro-debris/space dust, absorbing kinetic energy, damping rotation, or preventing secondary fragmentation on spacecraft.

Search Query Strategy:
Perform focused search queries combining both elements, such as:
- "magnetic fluid hypervelocity space debris impact"
- "self healing material spacecraft micrometeoroid kinetic energy"
- "magnetorheological fluid orbital debris shielding"
- "ferrofluid hypervelocity micro debris protection"

Output & Email Instructions:
- If you find genuinely relevant papers that discuss BOTH concepts and address this problem:
  1. Format your findings in clean Markdown starting with "YES":
     YES
     - Title:
     - Authors & Publication Date:
     - Source / URL Link:
     - Key Findings:
     - Comparison to my idea and problem:
  2. Send this report as an email using the send_email tool:
     - From: athishprajwalgr006@gmail.com
     - To: godandghostofshinobi@gmail.com
     - Subject: Research Match Found: Magnetic / Self-Healing Space Debris Mitigation

- If NO relevant paper is found:
  - Output strictly: "NO"
  - Do NOT call the send_email tool. Do NOT send any email.

Do not invent papers, authors, or links. Rely solely on retrieved search results.
"""
    return template_2
   

# Wrapping prompt_1_result
from langchain_core.runnables import RunnableLambda
prompt_1_result_runnable = RunnableLambda(prompt_1_result)



In [30]:
# PROMPT - 2 EXECUTION
from langchain_core.runnables import RunnableLambda
from IPython.core import magic_arguments
from langchain import agents
from dotenv import load_dotenv
load_dotenv()
def prompt_2_result(template_2):
     
    # LLM Setup
    from langchain_mistralai import ChatMistralAI

    llm = ChatMistralAI(model="ministral-14b-2512", temperature=0.2, max_retries=5)

    # TAVILY SEARCH TOOL
    from langchain_tavily import TavilySearch
    search_tool = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced",       # better for technical queries
    include_domains=["arxiv.org", "ntrs.nasa.gov", "ui.adsabs.harvard.edu",
                     "sciencedirect.com", "ieeexplore.ieee.org",
                     "nature.com", "mdpi.com"],
    )

    
    # Importing resend and creating a tool decorator in langchain (SADASEND EMAIL TOOL)
    from dotenv import load_dotenv
    load_dotenv()
    import os, smtplib, ssl
    from email.message import EmailMessage
    from dotenv import load_dotenv
    from langchain.tools import tool
    load_dotenv()

    @tool
    def send_email(from_email: str, to_email: str, subject: str, markdown_content: str) -> str:
        """Send the research report by email. Rejects articles that were already emailed."""
        papers = extract_papers(markdown_content)
        if not papers:
            return ("REJECTED: no articles found. Write each article with lines like "
                    "'Title: ...' and 'URL: https://...'.")

        new = get_new_papers(papers)
        dupes = [p["title"] for p in papers if p not in new]
        if dupes:
            return ("REJECTED: already emailed: " + "; ".join(dupes) +
                    ". Replace ONLY these with new articles and call send_email again "
                    "with the full report.")

        user = os.getenv("GMAIL_USER")
        pw = (os.getenv("GMAIL_APP_PASSWORD") or "").replace(" ", "")
        if not user or not pw:
            return "Failed: GMAIL_USER or GMAIL_APP_PASSWORD not found in environment."

        msg = EmailMessage()
        msg["From"] = user
        msg["To"] = to_email
        msg["Subject"] = subject
        msg.set_content("Attached is the research news report.")
        msg.add_attachment(markdown_content.encode("utf-8"), maintype="text",
                        subtype="plain", filename="Research_News.txt")
        try:
            with smtplib.SMTP_SSL("smtp.gmail.com", 465,
                                context=ssl.create_default_context(), timeout=20) as s:
                s.login(user, pw)
                s.send_message(msg)
        except Exception as e:
            return f"Failed to send email: {e}"

        mark_papers_sent(papers)          # saved only after a successful send
        return f"Email sent. {len(papers)} articles recorded."
    Tool_kit = [search_tool,send_email]
    
    #Agent Creation

    from langchain.agents import create_agent
    from langchain_ollama import ChatOllama

    Tool_kit = [search_tool,send_email]
    agent = create_agent(model=llm, tools=Tool_kit)


    from IPython.display import display, Markdown

    # --- OUTPUT 2: The Research News ---
    print("Running Prompt 2: Collecting Research News for my Research Idea...")
    result_2 = agent.invoke({"messages": [{"role": "user", "content": template_2}]})

    # --- debug: show what the agent really did ---
    for m in result_2["messages"]:
        for c in getattr(m, "tool_calls", None) or []:
            print("CALL:", c["name"])
        if m.type == "tool":
            print("TOOL RESULT:", m.name, "->", str(m.content)[:300])
    # ---------------------------------------------

    output_2 = Markdown(result_2["messages"][-1].content)

# Wrapping prompt_2_result
from langchain_core.runnables import RunnableLambda
prompt_2_result_runnable = RunnableLambda(prompt_2_result)


In [31]:
final_chain = prompt_1_runnable | prompt_1_result_runnable | prompt_2_result_runnable
response = final_chain.invoke({})



Running Prompt 1: Collecting Research News...
CALL: tavily_search
CALL: tavily_search
TOOL RESULT: tavily_search -> {"query": "latest scientific research news in Space astronomy novel discoveries subatomic particles curious phenomena last 30 days", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.kotatv.com/2026/10/24/nobel-prize-winning-discovery-connects-south-dakota-m
TOOL RESULT: tavily_search -> {"query": "groundbreaking scientific research news in Physics Particle Science Materials Science Nanotechnology Robotics Engineering Artificial Intelligence Machine Learning Chemistry Energy Healthcare Medicine last 30 days", "follow_up_questions": null, "answer": null, "images": [], "results": [{"u
CALL: tavily_search
CALL: tavily_search
TOOL RESULT: tavily_search -> {"query": "latest scientific breakthroughs in Space astronomy subatomic particles neutrinos dark matter black holes gravitational waves 2026", "follow_up_questions": null, "answer": 

Done

Running Prompt 2: Collecting Research News for my Research Idea...
CALL: tavily_search
CALL: tavily_search
CALL: tavily_search
CALL: tavily_search
TOOL RESULT: tavily_search -> {"query": "magnetic fluid hypervelocity space debris impact", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.sciencedirect.com/science/article/abs/pii/S1270963822003200", "title": "Impact resistance mechanism of reactive material bumper ...", "content": "
TOOL RESULT: tavily_search -> {"query": "self healing material spacecraft micrometeoroid kinetic energy", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.sciencedirect.com/topics/engineering/micrometeoroid", "title": "Micrometeoroid - an overview | ScienceDirect Topics", "content": "Th
TOOL RESULT: tavily_search -> {"query": "magnetorheological fluid orbital debris shielding", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://ntrs.nasa.

password length: 16 (should be 16)
FAILED: SMTPServerDisconnected Connection unexpectedly closed


SENT. Check godandghostofshinobi@gmail.com (and spam).
